# Code mẫu — Vision Transformer

ViT cắt ảnh thành các ô vuông nhỏ gọi là patch, biến mỗi patch thành một token, rồi cho các token trao đổi thông tin bằng attention. Các ví dụ dưới đây dùng trọng số đặt bằng tay để theo dõi đường đi của dữ liệu. Ta không tải một ViT pretrained nào cả.

## Ví dụ 1: Ảnh 4×4 thành bốn token

Ảnh xám chứa các số từ 0 tới 15, xếp lần lượt theo hàng. Cắt thành patch 2×2 thì được bốn patch, mỗi patch gồm đúng bốn ô kề nhau. Sau đó một phép chiếu tuyến tính biến mỗi patch thành một token ba chiều: chiều đầu là ô thứ nhất, chiều hai là ô thứ hai, chiều ba là trung bình của cả patch.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
image = torch.arange(16, dtype=torch.float32).reshape(1, 1, 4, 4)
windows = image.unfold(2, 2, 2).unfold(3, 2, 2)
# Đưa trục hàng và cột của patch lên trước các trục bên trong patch, rồi mới gộp.
patches = windows.permute(0, 2, 3, 1, 4, 5).reshape(1, 4, 4)
projection = nn.Linear(4, 3, bias=False)
with torch.no_grad():
    projection.weight.copy_(torch.tensor([[1., 0., 0., 0.],
                                          [0., 1., 0., 0.],
                                          [0.25, 0.25, 0.25, 0.25]]))
tokens = projection(patches)
print("Patch:", patches[0].int().tolist())
print("Token:", tokens[0].detach().tolist())
assert patches[0, 0].tolist() == [0, 1, 4, 5]

patch_conv = nn.Conv2d(1, 3, 2, stride=2, bias=False)
with torch.no_grad():
    patch_conv.weight.copy_(projection.weight.reshape(3, 1, 2, 2))
    via_conv = patch_conv(image).flatten(2).transpose(1, 2)
print("Linear trùng conv:", torch.allclose(tokens, via_conv))
assert torch.allclose(tokens, via_conv)

Patch: [[0, 1, 4, 5], [2, 3, 6, 7], [8, 9, 12, 13], [10, 11, 14, 15]]
Token: [[0.0, 1.0, 2.5], [2.0, 3.0, 4.5], [8.0, 9.0, 10.5], [10.0, 11.0, 12.5]]
Linear trùng conv: True


Patch đầu là `[0, 1, 4, 5]`, tức góc trên trái 2×2 của ảnh, chứ không phải bốn số đầu `[0, 1, 2, 3]` của hàng đầu. Nếu `reshape` thẳng mà không `permute`, ta sẽ nhận nhầm một hàng làm patch. Token đầu có trung bình là $(0+1+4+5)/4=2.5$.

Phần sau cho thấy một cách làm tương đương: conv với kernel 2 và stride 2. Mỗi vị trí của conv đọc đúng một patch, và dùng cùng trọng số với phép chiếu tuyến tính, nên cho đúng các token ấy. Các bản ViT thường cài bước cắt patch theo cách này.

## Ví dụ 2: Một query chọn đọc hai patch

Query `[1, 0]` được so với hai key `[1, 0]` và `[0, 1]`. Hai value mang thông tin 10 và 20. Điểm giữa query và từng key được chia cho căn của số chiều, rồi đi qua softmax theo trục key. Kết quả là các trọng số có tổng bằng 1, và đầu ra là trung bình có trọng số của các value.

In [ ]:
import math
import torch

torch.set_num_threads(1)
q = torch.tensor([[1., 0.]])
k = torch.tensor([[1., 0.], [0., 1.]])
v = torch.tensor([[10.], [20.]])
scores = q @ k.T / math.sqrt(k.shape[1])
weights = scores.softmax(dim=-1)
result = weights @ v
print("Điểm đã scale:", [[round(v, 4) for v in row] for row in scores.tolist()])
print("Trọng số:", [[round(v, 4) for v in row] for row in weights.tolist()])
print("Tổng trọng số:", weights.sum(-1).tolist())
print("Thông tin sau đọc:", round(result.item(), 4))
assert torch.allclose(weights.sum(-1), torch.ones(1))
assert 10 < result.item() < 20
assert weights[0, 0] > weights[0, 1]

Điểm đã scale: [[0.7071, 0.0]]
Trọng số: [[0.6698, 0.3302]]
Tổng trọng số: [1.0]
Thông tin sau đọc: 13.3024


Key đầu khớp với query hơn, nên nhận trọng số khoảng 0.67. Đầu ra là $0.6698\cdot10+0.3302\cdot20\approx13.30$, nằm giữa 10 và 20 và gần 10 hơn. Attention vẫn đọc cả hai value, chỉ là đọc patch này nhiều hơn patch kia. Trọng số attention cho biết token đã trộn thông tin ra sao. Nó không phải nhãn của vật thể, cũng không chứng minh vì sao model ra quyết định phân loại.

## Ví dụ 3: Đổi chỗ nội dung nhưng giữ nguyên vị trí

Bốn token hai chiều đại diện cho bốn patch. Ta cho chúng qua một lớp self-attention đơn giản, không dropout, rồi lấy trung bình mọi token. Sau đó so ba cách sắp xếp để thấy vai trò của position embedding, tức vector cộng thêm vào mỗi token để đánh dấu vị trí của nó.

In [ ]:
import math
import torch

torch.set_num_threads(1)
tokens = torch.tensor([[1., 0.], [0., 1.], [-1., 0.], [0., -1.]])
positions = torch.tensor([[0., 0.], [0., 0.4], [0.8, 0.], [0., -0.8]])
order = torch.tensor([2, 0, 3, 1])

def attend_mean(sequence):
    weights = (sequence @ sequence.T / math.sqrt(2)).softmax(-1)
    return (weights @ sequence).mean(0)

plain = attend_mean(tokens)
plain_shuffled = attend_mean(tokens[order])
positioned = attend_mean(tokens + positions)
content_moved = attend_mean(tokens[order] + positions)
both_moved = attend_mean((tokens + positions)[order])
print("Không position, đảo token vẫn trùng:", torch.allclose(plain, plain_shuffled, atol=1e-6))
print("Đảo cả nội dung và mã vị trí vẫn trùng:", torch.allclose(positioned, both_moved, atol=1e-6))
print("Chỉ đổi nội dung tại vị trí cố định còn trùng:",
      torch.allclose(positioned, content_moved, atol=1e-6))
print("Số ô attention, ảnh 4×4 rồi 8×8, patch 2:", 4**2, 16**2)
assert torch.allclose(plain, plain_shuffled, atol=1e-6)
assert torch.allclose(positioned, both_moved, atol=1e-6)
assert not torch.allclose(positioned, content_moved, atol=1e-6)

Không position, đảo token vẫn trùng: True
Đảo cả nội dung và mã vị trí vẫn trùng: True
Chỉ đổi nội dung tại vị trí cố định còn trùng: False
Số ô attention, ảnh 4×4 rồi 8×8, patch 2: 16 256


Không có position embedding, xáo thứ tự các patch không làm đổi kết quả. Model không biết patch nào nằm ở đâu, giống như nhìn một túi mảnh ghép lộn xộn. Khi xáo cả nội dung lẫn mã vị trí đi kèm, kết quả vẫn không đổi, vì mỗi patch vẫn mang đúng vị trí của nó. Chỉ khi giữ mã vị trí cố định và đổi chỗ nội dung, tức bố cục ảnh thật sự thay đổi, kết quả mới khác.

Dòng cuối là chi phí. Ảnh 4×4 với patch 2 cho 4 patch, nên bảng attention có 16 ô. Ảnh 8×8 cho 16 patch, nên bảng có 256 ô. Gấp đôi mỗi chiều của ảnh làm số patch gấp 4, và bảng attention gấp 16.